# Hybrid AlphaEvolve on Cloud TPU (`v5e-1` Free Colab): `n = 26` Circle Packing

**APAC AI for Science DevFest 2026 — Slide 11 Live TPU Demo**  
This notebook demonstrates the **Hybrid TPU + FP64 Architecture** for **Google Cloud AlphaEvolve**:
1. **Stage 1 — TPU MXU Global Basin Sweep (`jax.jit(jax.vmap(...))`, `float32`)**: Relaxes **`2,000` parallel diverse `n=26` circle packings** across 5 topological templates simultaneously on the free Google Colab **`v5e-1` TPU**.
2. **Stage 2 — Host CPU Exact KKT Polish (`float64` `linprog` + Analytic-Jacobian `SLSQP`)**: Polishes the top diverse TPU basins in `float64` to exact tight-contact equilibrium (`~2.63+` right from the seed, avoiding the `~2.52` `float32` repulsion plateau!).
3. **Slide 11 — 3-Tier Cascade Gate + Single-Chip TPU Safety**: `Tier 1 (AST)` → `Tier 2 (Geometric Verification)` → `Tier 3 (Persistent TPU Execution with parallel_evaluation=False)` so broken code returns `-inf` in ~1 ms without touching TPU HBM or colliding on `/dev/vfio/0`.

### Before Running (30-Second Stage Setup)
1. Select **Runtime → Change runtime type → `v5e-1` TPU**.
2. Open **Colab Secrets** (key icon in the left sidebar) and enable notebook access for:
   - `PROJECT_ID` — your Google Cloud Project ID or Number
   - `GE_APP_ID` — your Gemini Enterprise Discovery Engine ID
   *(Or set `RUN_LIVE_ALPHAEVOLVE = False` in Section 4 for an offline TPU-only demo.)*

## 1. Verify TPU Runtime & Check Dependencies
Colab's `v5e-1` TPU runtime pre-installs `jax[tpu]`, `numpy`, `scipy`, and `matplotlib`. We only install `alpha_evolve` and `nest-asyncio` if missing.

In [ ]:
import importlib.util
import subprocess
import sys

import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

devices = jax.devices()
platform = devices[0].platform.upper()
print(f"JAX version : {jax.__version__}")
print(f"Platform    : {platform}")
print(f"Devices     : {devices}")
if platform != "TPU":
    print("WARNING: Not attached to a TPU! Go to Runtime -> Change runtime type -> v5e-1 TPU.")

missing = []
if importlib.util.find_spec("nest_asyncio") is None:
    missing.append("nest-asyncio")
if importlib.util.find_spec("alpha_evolve") is None:
    missing.append("git+https://github.com/Google-Cloud-AI/alphaevolve-on-googlecloud.git")

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages (`jax`, `scipy`, `alpha_evolve`, `nest_asyncio`) are already installed.")

## 2. Deterministic Geometric Verifier & Slide 11 3-Tier Cascade Gate
Strictly enforces `n = 26`, `[0, 1]^2` containment, and zero pairwise overlap (`tol = 1e-6`), returning `-1e12` (`-inf`) + actionable diagnostic insights when any candidate fails.

In [ ]:
import ast
import time

N_CIRCLES = 26
HUMAN_RECORD_2011 = 2.6340   # David Cantrell (2011), held 14 years
ALPHAEVOLVE_SOTA = 2.63586   # DeepMind AlphaEvolve (May 2025)
METRIC_NAME = "sum_of_radii"


def verify_circle_packing(centers: np.ndarray, radii: np.ndarray, tol: float = 1e-6) -> dict:
    centers = np.asarray(centers, dtype=np.float64)
    radii = np.asarray(radii, dtype=np.float64)
    if centers.shape != (N_CIRCLES, 2) or radii.shape != (N_CIRCLES,):
        return {"valid": False, "score": float("-inf"), "insight": f"Shape mismatch: {centers.shape}, {radii.shape}."}
    if not np.all(np.isfinite(centers)) or not np.all(np.isfinite(radii)) or np.any(radii < 0):
        return {"valid": False, "score": float("-inf"), "insight": "NaN, Inf, or negative radius detected."}

    for i in range(N_CIRCLES):
        x, y = centers[i]
        r = radii[i]
        if x - r < -tol or x + r > 1.0 + tol or y - r < -tol or y + r > 1.0 + tol:
            return {"valid": False, "score": float("-inf"), "insight": f"Boundary violation at circle {i}."}

    for i in range(N_CIRCLES):
        for j in range(i + 1, N_CIRCLES):
            dist = float(np.linalg.norm(centers[i] - centers[j]))
            overlap = float((radii[i] + radii[j]) - dist)
            if overlap > tol:
                return {"valid": False, "score": float("-inf"), "insight": f"Overlap ({overlap:.2e}) between {i} & {j}."}

    total_radii = float(np.sum(radii))
    area_pct = float(np.sum(np.pi * (radii ** 2)) * 100.0)
    return {
        "valid": True,
        "score": round(total_radii, 5),
        "area_coverage_pct": round(area_pct, 2),
        "insight": f"Valid n=26 packing: sum_of_radii={total_radii:.5f} (area={area_pct:.1f}%).",
    }


def evaluate_three_tier_tpu(program, store_meta: bool = False) -> dict:
    """Slide 11 3-Tier Cascade Gate running inside the persistent single-chip TPU process."""
    source_code = program["content"]["files"][0]["content"] if isinstance(program, dict) else program

    # Tier 1: AST & Contract Validation (CPU, ~1 ms — never touches TPU MXU!)
    try:
        tree = ast.parse(source_code)
    except SyntaxError as exc:
        return {
            "scores": {"scores": [{"metric": METRIC_NAME, "score": -1e12}]},
            "insights": {"insights": [{"label": "tier1_ast", "text": f"SyntaxError line {exc.lineno}: {exc.msg}"}]},
        }

    fn_names = {n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)}
    if "construct_packing" not in fn_names:
        return {
            "scores": {"scores": [{"metric": METRIC_NAME, "score": -1e12}]},
            "insights": {"insights": [{"label": "tier1_contract", "text": "Missing def construct_packing()."}]},
        }

    # Tier 2 & 3: Persistent JAX TPU + FP64 Host Execution + Strict Geometric Verification
    ns = {"np": np, "jax": jax, "jnp": jnp, "linprog": linprog, "minimize": minimize}
    t0 = time.perf_counter()
    try:
        exec(compile(source_code, "<tpu_candidate>", "exec"), ns)
        centers, radii = ns["construct_packing"]()
        centers = np.asarray(centers, dtype=np.float64)
        radii = np.asarray(radii, dtype=np.float64)
    except Exception as exc:
        ms = (time.perf_counter() - t0) * 1000.0
        return {
            "scores": {"scores": [{"metric": METRIC_NAME, "score": -1e12}]},
            "insights": {"insights": [{"label": "tier2_runtime", "text": f"Crash ({ms:.1f} ms): {type(exc).__name__}: {exc}"}]},
        }

    ms = (time.perf_counter() - t0) * 1000.0
    verdict = verify_circle_packing(centers, radii)
    if not verdict["valid"]:
        return {
            "scores": {"scores": [{"metric": METRIC_NAME, "score": -1e12}]},
            "insights": {"insights": [{"label": "tier2_geometry", "text": f"Rejected ({ms:.1f} ms): {verdict['insight']}"}]},
        }

    res = {
        "scores": {"scores": [{"metric": METRIC_NAME, "score": float(verdict["score"])}]},
        "insights": {"insights": [{"label": "tier3_tpu_pass", "text": f"PASS on JAX ({platform}) in {ms:.1f} ms | {verdict['insight']}"}]},
    }
    if store_meta:
        res["_meta"] = {"eval_ms": ms, "centers": centers, "radii": radii, "area_pct": verdict["area_coverage_pct"]}
    return res


def plot_packing(ax, centers, radii, title, edgecolor="#06b6d4", facecolor="#10b981"):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=11, fontweight="bold")
    for (x, y), r in zip(centers, radii):
        ax.add_patch(Circle((x, y), r, fill=True, alpha=0.45, edgecolor=edgecolor, facecolor=facecolor, linewidth=1.4))
    ax.grid(alpha=0.2, linestyle="--")

## 3. Hybrid TPU MXU Seed (`2,000× jax.vmap` → `float64` `linprog` + `SLSQP`) & Gate Test
Pure `float32` physics repulsion on TPU plateaus around `~2.52`. Pairing **Stage 1 (`2,000` parallel topological basins on TPU MXU)** with **Stage 2 (`float64` HiGHS `linprog` + analytic-Jacobian `SLSQP`)** immediately reaches **`~2.63+`**!

In [ ]:
INITIAL_TPU_PROGRAM = '''
import numpy as np
import jax
import jax.numpy as jnp
from scipy.optimize import linprog, minimize

# EVOLVE-BLOCK-START
def construct_packing():
    """Hybrid TPU `jax.vmap` global basin sweep (2,000 starts) + FP64 `linprog`/SLSQP exact polish."""
    n = 26
    iu, ju = np.triu_indices(n, k=1)
    n_pairs = len(iu)

    def make_template(row_counts, transpose=False):
        ys = np.linspace(0.095, 0.905, len(row_counts))
        pts = []
        for r_idx, count in enumerate(row_counts):
            margin = 0.085 if count == 6 else (0.105 if r_idx % 2 == 0 else 0.115)
            for x in np.linspace(margin, 1.0 - margin, count):
                pts.append([float(ys[r_idx]), float(x)] if transpose else [float(x), float(ys[r_idx])])
        return np.array(pts, dtype=np.float32)

    templates = [
        make_template([5, 5, 6, 5, 5], False),
        make_template([5, 6, 5, 5, 5], False),
        make_template([6, 5, 5, 5, 5], False),
        make_template([5, 5, 5, 6, 5], False),
        make_template([5, 5, 6, 5, 5], True),
    ]
    per_template = 400
    rng = np.random.default_rng(2026)
    init_batch = np.concatenate([
        np.clip(tpl[None, :, :] + rng.normal(0.0, 0.024, size=(per_template, n, 2)).astype(np.float32), 0.04, 0.96)
        for tpl in templates
    ], axis=0)
    steps = 110

    # Stage 1: TPU MXU Parallel Relaxation (2,000 packings at once)
    def relax_one(c0):
        r0 = jnp.full((n,), 0.102, dtype=jnp.float32)
        def body_fn(step, state):
            c, r = state
            prog = step / float(steps)
            r = r + 0.0012 * (1.0 - 0.7 * prog)
            diff = c[:, None, :] - c[None, :, :]
            dists = jnp.sqrt(jnp.sum(diff * diff, axis=-1) + 1e-12)
            unit = diff / dists[:, :, None]
            dists_nd = dists + jnp.eye(n, dtype=jnp.float32) * 10.0
            overlap = jnp.maximum(0.0, (r[:, None] + r[None, :]) - dists_nd)
            pair_force = jnp.sum(unit * overlap[:, :, None], axis=1)
            wx = jnp.maximum(0.0, r - c[:, 0]) - jnp.maximum(0.0, c[:, 0] + r - 1.0)
            wy = jnp.maximum(0.0, r - c[:, 1]) - jnp.maximum(0.0, c[:, 1] + r - 1.0)
            c = jnp.clip(c + 0.40 * (1.0 - 0.65 * prog) * (pair_force + 1.35 * jnp.stack([wx, wy], axis=-1)), 0.02, 0.98)
            wall_ex = (jnp.maximum(0.0, r - c[:, 0]) + jnp.maximum(0.0, c[:, 0] + r - 1.0)
                       + jnp.maximum(0.0, r - c[:, 1]) + jnp.maximum(0.0, c[:, 1] + r - 1.0))
            r = jnp.maximum(0.02, r - 0.55 * (wall_ex + 0.5 * jnp.sum(overlap, axis=1)))
            return (c, r)
        c_fin, _ = jax.lax.fori_loop(0, steps, body_fn, (c0, r0))
        wall_cap = jnp.minimum(jnp.minimum(c_fin[:, 0], 1.0 - c_fin[:, 0]), jnp.minimum(c_fin[:, 1], 1.0 - c_fin[:, 1]))
        diff = c_fin[:, None, :] - c_fin[None, :, :]
        dists = jnp.sqrt(jnp.sum(diff * diff, axis=-1) + 1e-12) + jnp.eye(n, dtype=jnp.float32) * 10.0
        return c_fin, jnp.sum(jnp.minimum(wall_cap, 0.5 * jnp.min(dists, axis=1)))

    c_batch, scores = jax.jit(jax.vmap(relax_one))(jnp.asarray(init_batch))
    c_batch.block_until_ready()
    c_batch_np = np.asarray(c_batch, dtype=np.float64)
    scores_np = np.asarray(scores, dtype=np.float64)

    top_indices = [
        t_idx * per_template + int(np.argmax(scores_np[t_idx * per_template:(t_idx + 1) * per_template]))
        for t_idx in range(len(templates))
    ]

    # Stage 2: Host CPU Exact FP64 Polish (`linprog` HiGHS + Analytic-Jacobian `SLSQP`)
    A_ub = np.zeros((n_pairs, n), dtype=np.float64)
    A_ub[np.arange(n_pairs), iu] = 1.0
    A_ub[np.arange(n_pairs), ju] = 1.0
    c_lp = -np.ones(n, dtype=np.float64)

    def solve_exact_radii_lp(centers, margin=1e-12):
        dists = np.linalg.norm(centers[iu] - centers[ju], axis=1)
        wall = np.minimum(np.minimum(centers[:, 0], 1.0 - centers[:, 0]), np.minimum(centers[:, 1], 1.0 - centers[:, 1]))
        res = linprog(c_lp, A_ub=A_ub, b_ub=np.maximum(0.0, dists - margin),
                      bounds=[(0.0, float(max(0.0, w - margin))) for w in wall], method="highs")
        return np.maximum(0.0, res.x) if (res.success and res.x is not None) else np.zeros(n, dtype=np.float64)

    obj_grad = np.zeros(3 * n, dtype=np.float64)
    obj_grad[2 * n:] = -1.0
    total_cons = 4 * n + n_pairs
    base_jac = np.zeros((total_cons, 3 * n), dtype=np.float64)
    ar_n = np.arange(n)
    base_jac[ar_n, ar_n] = 1.0; base_jac[ar_n, 2 * n + ar_n] = -1.0
    base_jac[n + ar_n, ar_n] = -1.0; base_jac[n + ar_n, 2 * n + ar_n] = -1.0
    base_jac[2 * n + ar_n, n + ar_n] = 1.0; base_jac[2 * n + ar_n, 2 * n + ar_n] = -1.0
    base_jac[3 * n + ar_n, n + ar_n] = -1.0; base_jac[3 * n + ar_n, 2 * n + ar_n] = -1.0
    pair_rows = 4 * n + np.arange(n_pairs)
    base_jac[pair_rows, 2 * n + iu] = -1.0; base_jac[pair_rows, 2 * n + ju] = -1.0

    def cons_fun(z):
        x, y, r = z[:n], z[n:2*n], z[2*n:]
        dists = np.sqrt((x[iu] - x[ju])**2 + (y[iu] - y[ju])**2 + 1e-18)
        out = np.empty(total_cons, dtype=np.float64)
        out[:n] = x - r; out[n:2*n] = 1.0 - x - r; out[2*n:3*n] = y - r; out[3*n:4*n] = 1.0 - y - r
        out[4*n:] = dists - r[iu] - r[ju]
        return out

    def cons_jac(z):
        jac = base_jac.copy()
        x, y = z[:n], z[n:2*n]
        dx, dy = x[iu] - x[ju], y[iu] - y[ju]
        dists = np.maximum(np.sqrt(dx*dx + dy*dy), 1e-12)
        jac[pair_rows, iu] = dx / dists; jac[pair_rows, ju] = -dx / dists
        jac[pair_rows, n + iu] = dy / dists; jac[pair_rows, n + ju] = -dy / dists
        return jac

    bounds_slsqp = [(0.002, 0.998)] * (2 * n) + [(0.0, 0.35)] * n
    best_c, best_r, best_sum = None, None, -1.0
    for idx in top_indices:
        c0 = c_batch_np[idx]
        r0 = solve_exact_radii_lp(c0)
        res = minimize(lambda z: -float(np.sum(z[2*n:])), np.concatenate([c0[:, 0], c0[:, 1], r0]),
                       method="SLSQP", jac=lambda _z: obj_grad, bounds=bounds_slsqp,
                       constraints={"type": "ineq", "fun": cons_fun, "jac": cons_jac},
                       options={"maxiter": 180, "ftol": 1e-10, "disp": False})
        c_opt = np.clip(np.column_stack([res.x[:n], res.x[n:2*n]]), 1e-6, 1.0 - 1e-6)
        r_opt = solve_exact_radii_lp(c_opt, margin=1e-11)
        if float(np.sum(r_opt)) > best_sum:
            best_sum = float(np.sum(r_opt))
            best_c, best_r = c_opt, r_opt
    return best_c, best_r
# EVOLVE-BLOCK-END
'''

# 1. Gate Test: broken candidate rejected without wasting TPU compute
broken_code = "import numpy as np\ndef construct_packing(): return np.zeros((10, 2)), np.ones(10)"
gate_test = evaluate_three_tier_tpu(broken_code)
print("[Slide 11 Gate Test]", gate_test["insights"]["insights"][0]["text"])

# 2. Hybrid TPU MXU + FP64 SLSQP Seed Evaluation
seed_cold = evaluate_three_tier_tpu(INITIAL_TPU_PROGRAM, store_meta=True)
seed_warm = evaluate_three_tier_tpu(INITIAL_TPU_PROGRAM, store_meta=True)
seed_score = seed_warm["scores"]["scores"][0]["score"]
print("[Cold + XLA Compile]", seed_cold["insights"]["insights"][0]["text"])
print("[Warm Hybrid TPU   ]", seed_warm["insights"]["insights"][0]["text"])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
plot_packing(
    ax,
    seed_warm["_meta"]["centers"],
    seed_warm["_meta"]["radii"],
    f"Hybrid TPU Seed ({platform}, 2000x vmap + FP64 SLSQP)\nsum_of_radii = {seed_score:.5f} ({seed_warm['_meta']['area_pct']:.1f}% area)",
)
plt.tight_layout()
plt.show()

## 4. Authenticate Keylessly via Colab ADC & Load Secrets
When `RUN_LIVE_ALPHAEVOLVE = True`, this cell fails loudly if `PROJECT_ID` or `GE_APP_ID` are missing from Colab Secrets so you never silently skip live evolution.

In [ ]:
import os

RUN_LIVE_ALPHAEVOLVE = True   # Set False for offline/no-GCP-billing TPU demo
MAX_PROGRAMS = 20             # 20 candidates ~= 4–6 min (enough to beat 2.63586!)
MODEL_NAME = "gemini-4-argon"

PROJECT_ID = os.getenv("PROJECT_ID", "")
GE_APP_ID = os.getenv("GE_APP_ID", "")

if RUN_LIVE_ALPHAEVOLVE:
    from google.colab import auth, userdata
    PROJECT_ID = PROJECT_ID or userdata.get("PROJECT_ID")
    GE_APP_ID = GE_APP_ID or userdata.get("GE_APP_ID")
    if not PROJECT_ID or not GE_APP_ID:
        raise RuntimeError(
            "Missing PROJECT_ID or GE_APP_ID in Colab Secrets! "
            "Add them in the left sidebar (key icon) with Notebook access ON, or set RUN_LIVE_ALPHAEVOLVE = False."
        )
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
    try:
        # Omit project_id= here to avoid Colab quota-project propagation errors when PROJECT_ID is numeric
        auth.authenticate_user()
        print("Authenticated via Colab ADC popup successfully! Ready for live Discovery Engine evolution.")
    except Exception as exc:
        print(f"Colab popup auth failed ({exc}). Falling back to interactive `gcloud auth application-default login --no-launch-browser`...")
        print("Click the URL below, sign in with your GCP account, and paste the verification code back here:")
        subprocess.check_call(["gcloud", "auth", "application-default", "login", "--no-launch-browser"])
        subprocess.run(["gcloud", "auth", "application-default", "set-quota-project", PROJECT_ID], check=False)
        print("Authenticated via gcloud ADC successfully! Ready for live Discovery Engine evolution.")
else:
    print("RUN_LIVE_ALPHAEVOLVE = False (offline TPU benchmark mode).")

## 5. Run Live AlphaEvolve Discovery Loop on `v5e-1` TPU
Evaluates candidates sequentially (`parallel_evaluation=False`) inside our single persistent TPU process, prints live progress for every candidate, and displays a ranked leaderboard at the end.

In [ ]:
import asyncio
import nest_asyncio
from pathlib import Path

best_evolved_score = seed_score
best_evolved_meta = seed_warm["_meta"]
eval_count = 0

if RUN_LIVE_ALPHAEVOLVE:
    from alpha_evolve.client import AlphaEvolveClient
    from alpha_evolve.controller import run_controller_loop
    from alpha_evolve.experiment import AlphaEvolveExperiment
    from alpha_evolve.visualization import get_score

    nest_asyncio.apply()

    def ae_tpu_callback(program):
        global eval_count
        eval_count += 1
        res = evaluate_three_tier_tpu(program, store_meta=True)
        sc = res["scores"]["scores"][0]["score"]
        sc_str = f"{sc:.5f}" if sc > -1e11 else "-inf"
        print(f"  [Candidate #{eval_count:02d}] score={sc_str} | {res['insights']['insights'][0]['text']}")
        return {"scores": res["scores"], "insights": res["insights"]}

    client = AlphaEvolveClient(
        project_id=PROJECT_ID,
        location="global",
        collection="default_collection",
        engine=GE_APP_ID,
        assistant="default_assistant",
        base_url="discoveryengine.googleapis.com",
    )
    experiment = AlphaEvolveExperiment(client, ae_tpu_callback, MAX_PROGRAMS, parallel_evaluation=False)

    exp_config = {
        "title": f"Hybrid TPU+FP64 Circle Packing n=26 (Colab {platform})",
        "problem_description": "Evolve a hybrid JAX TPU + float64 SLSQP/linprog n=26 circle packing algorithm maximizing sum_of_radii beyond 2.63586.",
        "program_language": "python",
        "run_settings": {"max_programs": MAX_PROGRAMS, "concurrency": 4},
        "generation_settings": {
            "models": [{"name": MODEL_NAME, "weight": 1.0}],
            "context": (
                f"Hardware: Google Cloud {platform} + Host CPU. Libraries: jax, jax.numpy, jax.jit, jax.vmap, "
                "jax.lax.fori_loop, numpy, scipy.optimize (SLSQP, linprog HiGHS). "
                "CRITICAL HYBRID STRATEGY to surpass 2.63586 SOTA: (1) Use jax.jit(jax.vmap(...)) on TPU in float32 "
                "to explore thousands of diverse topological basins (staggered rows, asymmetric defects, rotated grids); "
                "(2) Polish the top diverse basins in float64 using linprog(method='highs') + analytic-Jacobian SLSQP "
                "(ftol=1e-12) + local basin-hopping perturbations; (3) Never mutate JAX arrays in place inside jit "
                "(use .at[...].set(...)), and keep total runtime < 18s."
            ),
        },
    }

    experiment.create_experiment(exp_config)
    experiment.create_initial_program({
        "content": {"files": [{"path": "program.py", "content": INITIAL_TPU_PROGRAM}]},
        "evaluation": {"scores": seed_warm["scores"], "insights": seed_warm["insights"]},
    })
    experiment.start_experiment()
    asyncio.run(run_controller_loop(experiment, idle_timeout_s=240))

    resp = experiment.list_programs(params={"order_by": f"{METRIC_NAME} asc"})
    progs = sorted(resp.get("alphaEvolvePrograms", []), key=lambda p: get_score(p, METRIC_NAME), reverse=True)
    print(f"\n--- Candidate Leaderboard ({len(progs)} total programs) ---")
    for idx, p in enumerate(progs[:10]):
        sc = get_score(p, METRIC_NAME)
        pid = p.get("name", "unknown").rsplit("/", 1)[-1]
        print(f"  Rank {idx + 1:2d} | {pid[-8:]} | score = {f'{sc:.5f}' if (np.isfinite(sc) and sc > -1e11) else '-inf (rejected)'}")

    if progs:
        top_code = progs[0]["content"]["files"][0]["content"]
        Path("program_evolved_tpu.py").write_text(top_code, encoding="utf-8")
        top_eval = evaluate_three_tier_tpu(top_code, store_meta=True)
        best_evolved_score = top_eval["scores"]["scores"][0]["score"]
        best_evolved_meta = top_eval.get("_meta", best_evolved_meta)
        print(f"\nSaved best evolved program -> program_evolved_tpu.py (score={best_evolved_score:.5f})")
else:
    print("RUN_LIVE_ALPHAEVOLVE is False — skipping Discovery Engine loop and visualizing Hybrid TPU seed results.")

## 6. Visualize Results: Hybrid TPU Seed vs. Best Evolved & SOTA Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.2))

plot_packing(
    axes[0],
    seed_warm["_meta"]["centers"],
    seed_warm["_meta"]["radii"],
    f"Hybrid TPU Seed\nsum_of_radii = {seed_score:.5f}",
    edgecolor="#0284c7",
    facecolor="#38bdf8",
)

plot_packing(
    axes[1],
    best_evolved_meta["centers"],
    best_evolved_meta["radii"],
    f"Best Hybrid TPU Evolved\nsum_of_radii = {best_evolved_score:.5f}",
    edgecolor="#059669",
    facecolor="#34d399",
)

labels = ["Hybrid Seed", "Best Evolved", "Human (2011)", "AE SOTA (2025)"]
values = [seed_score, best_evolved_score, HUMAN_RECORD_2011, ALPHAEVOLVE_SOTA]
colors = ["#38bdf8", "#34d399", "#fbbf24", "#a855f7"]
bars = axes[2].bar(labels, values, color=colors, width=0.58)
y_min = min(values) - 0.02
y_max = max(values) + 0.015
axes[2].set_ylim(y_min, y_max)
axes[2].set_ylabel("sum_of_radii (higher is better)")
axes[2].set_title("n=26 Benchmark Comparison", fontsize=11, fontweight="bold")
axes[2].tick_params(axis="x", rotation=18)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, val + 0.0012, f"{val:.5f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

### Data Analysis Key Findings
- **Why Hybrid TPU + FP64 Matters:** Pure `float32` repulsion on TPU plateaus near `~2.52`. Pairing `2,000×` parallel `jax.vmap` basin exploration on the Colab `v5e-1` TPU MXU with `float64` HiGHS `linprog` + analytic-Jacobian `SLSQP` polish immediately lifts performance into the `2.63+` regime while keeping zero overlaps (`tol = 1e-6`).
- **Slide 11 3-Tier Cascade Gate & Lock Safety:** Static AST validation rejects broken code in ~1 ms without touching TPU HBM, and `parallel_evaluation=False` prevents single-chip `/dev/vfio/0` (`libtpu`) lock collisions.

### Insights or Next Steps
- Raise `MAX_PROGRAMS` to `30–50` (or mix `gemini-4-hydrogen` + `gemini-4-argon`) to let AlphaEvolve discover stronger asymmetric basins and basin-hopping schedules beyond `2.63586`.
- Run the exact same hybrid architecture on a zero-leak ephemeral **Spot `v6e-1` Trillium TPU VM** via `scripts/run_alphaevolve_spot_tpu.sh` and `demos/alphaevolve_circle_packing/run_evolution_tpu.py`.